# Foundations of Agents: From First Principles to the OpenAI Agents SDK


## What we are doing today


This course starts small and adds capability one layer at a time.

```mermaid
flowchart LR
    A[Single LLM Call] --> B[Chatbot]
    B --> C[Structured Outputs]
    C --> D[Actions / Tools]
    D --> E[Agent Loop]
    E --> G[OpenAI Agents SDK]
```


## Enviroment Setup

Before running the code, install the required packages:

```bash
pip install openai rich python-dotenv
```

You also need an OpenAI API key available as an environment variable.

On macOS or Linux:

```bash
export OPENAI_API_KEY="your-api-key"
```

On Windows PowerShell:

```powershell
setx OPENAI_API_KEY "your-api-key"
```

In [1]:
# loading enviroment variables

from dotenv import load_dotenv
from pathlib import Path
import os

project_root = Path.cwd()

while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

env_path = project_root / ".env"

print(".env exists:", env_path.exists())

load_dotenv(env_path)

.env exists: True


True

In [2]:
from __future__ import annotations

import os

from openai import OpenAI
from rich import print
from rich.panel import Panel

---
---


# Chapter 1 — From an LLM Call to a Chatbot

## Learning objectives


By the end of this chapter, you will be able to:

- Make a simple LLM call using the OpenAI Responses API.
- Explain why an LLM call is stateless.
- Understand how prompts shape model behaviour.
- Explain why conversation history is needed.
- Build a simple chatbot using structured messages.


In this chapter, we start with the smallest possible AI application: one request to a model.

Then we gradually add the missing pieces needed for a chatbot:

```mermaid
flowchart LR
    A[Single LLM Call] --> B[Prompting]
    B --> C[Statelessness]
    C --> D[Conversation History]
    D --> E[Structured Messages]
    E --> F[Chatbot]

## 1. The smallestLLM application: one LLM call

#### Let's start with a question

Before writing code, let's ask the simplest possible question:

> **What is the smallest useful AI application we can build with LLMs?**

A reasonable answer is:

> A program that sends a request to an LLM and prints the response.

No chatbot.  
No tools.  
No memory.  
No agent.

Just one input and one output.

```mermaid
flowchart LR
    U[User input] --> L[LLM]
    L --> R[Text response]
```

💡 **Key Idea**

An LLM is not magic from the perspective of your Python programme. It is an API that receives input and returns output.

#### Create the client


The `OpenAI` client is the object we use to talk to the LLM Model (via API call).

We also define the model name once, so the rest of the notebook can refer to it consistently.

In [3]:
MODEL = os.getenv("MODEL","gpt-4o-mini")
api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

#### Make the first LLM call


Let's make one simple request.

This is intentionally plain. At this stage we want to understand the shape of the API call.

In [4]:
response = client.responses.create(
    model=MODEL,
    input="Explain what an AI agent is in one short paragraph.",
)

print(response.output_text)

An AI agent is a software entity that autonomously perceives its environment and takes actions to achieve specific 
goals. Equipped with algorithms and machine learning capabilities, it can analyze data, make decisions, and adapt 
to changing conditions. AI agents are used in various applications, from virtual assistants and customer service 
bots to robotic systems and autonomous vehicles, enabling them to operate intelligently in real-world scenarios.

#### What just happened?


The call had three important parts:

| Part | Meaning |
|---|---|
| `model` | Which model should respond |
| `input` | What we want the model to respond to |
| `response.output_text` | The generated text returned by the model |

At this point, our application has no memory and no special behaviour. It simply sends a request and prints the result.

🤔 **Think About It**

If we ran this cell again, would we always get exactly the same wording?

Probably not. LLMs can produce slightly different outputs even for similar inputs. This is one reason we need to design applications carefully when reliability matters.

#### Wrap the call in a small function



We are going to call the model several times.

Instead of repeating the same API code, let's wrap it in a function.

This is not a heavy abstraction. It is just good Python hygiene.

In [30]:
def call_llm(prompt: str, model: str = MODEL) -> str:
    """Send a single prompt to the model and return the text response.
    """
    
    response = client.responses.create(model=model,input=prompt)
    return response.output_text

In [31]:
answer = call_llm("Explain prompt engineering in two sentences.")
print(answer)

Prompt engineering involves crafting specific and effective input prompts to guide AI models in generating desired 
outputs. By carefully selecting wording and structure, users can enhance the relevance and quality of the responses
provided by the model.

✅ **Best Practice**

Keep helper functions small and obvious.

This function does one thing:

> Send text to the model and return text back.

Later, as our application becomes more capable, we will add new patterns. For now, simple is better.

## 2. Prompting shapes behaviour

The model does not only respond to the topic of the prompt.

It also responds to the style, audience, constraints and role implied by the prompt.

#### Same Question, different ways

Let's ask the same broad question in two different ways.

The model is the same.  
The API call is the same.  
Only the prompt changes.

In [33]:
vague_answer = call_llm("Explain APIs. keep your response breif.")

specific_answer = call_llm(
    "Explain APIs to an intermediate Python developer. "
    "Use one practical example and avoid unnecessary jargon."
    "keep your response short."
)

print(Panel(vague_answer, title="Vague prompt"))
print(Panel(specific_answer, title="Specific prompt"))

╭───────────────────────────────────────────────── Vague prompt ──────────────────────────────────────────────────╮
│ APIs, or Application Programming Interfaces, are sets of rules and protocols that allow different software      │
│ applications to communicate with each other. They define the methods and data formats that applications can use │
│ to request and exchange information. APIs enable developers to integrate different services, access features of │
│ operating systems, or connect to databases, all without needing to understand the underlying code. This         │
│ promotes interoperability and can enhance functionality in software development.                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Specific prompt ────────────────────────────────────────────────╮
│ APIs (Application Programming Interfaces) allow different software applications to communicate with each other. │
│ Think of an API as a menu in a restaurant. It lists the options available (endpoints), and when you place an    │
│ order (send a request), the kitchen (the server) prepares the meal and sends it back to you (the response).     │
│                                                                                                                 │
│ ### Practical Example: Fetching Weather Data                                                                    │
│                                                                                                                 │
│ Imagine you want to develop a Python app that shows the current weather. Instead of building the entire weather │
│ system, you can use a weather API like OpenWeatherMap. Here’s a simple example of how to get the current        │
│ weather:                                                                                                        │
│                                                                                                                 │
│ 1. **Sign up** for an API key from OpenWeatherMap.                                                              │
│ 2. **Use requests** to make a call to the API.                                                                  │
│                                                                                                                 │
│ ```python                                                                                                       │
│ import requests                                                                                                 │
│                                                                                                                 │
│ # Your API key                                                                                                  │
│ api_key = "your_api_key"                                                                                        │
│ city = "London"                                                                                                 │
│ url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"                   │
│                                                                                                                 │
│ # Making a request to the API                                                                                   │
│ response = requests.get(url)                                                                                    │
│                                                                                                                 │
│ # Checking if the request was successful                                                                        │
│ if response.status_code == 200:                                                                                 │
│     data = response.json()  # Convert the response to JSON                                                      │
│     temperature = data['main']['temp']  # Extract the temperature                                               │
│     print(f"The temperature in {city} is {temperature}°C.")                                                     │
│ else:                                                                                                           │
│     print("Error:", response.status_code)                                                                       │
│ ```                                                                                                             │
│                                                                                                                 │
│ ### Key Points:                                       

#### What changed?


The model did not gain new information between the two calls.

What changed was the **context** we gave it.

The second prompt specified:

- the audience
- the desired style
- the level of detail
- the kind of example we wanted

🤔 **Think About It**

Which response would be more useful in a classroom? Why?

💡 **Key Idea**

Prompting is not just asking a question. Prompting is **designing the context** that guides the model's behaviour.

#### A useful mental model


For now, think of the LLM call like this:

```mermaid
flowchart LR
    A[Prompt] --> B[Model]
    B --> C[Response]
```

But the prompt is doing more than asking a question.

It can include:

- the task
- the audience
- the tone
- constraints
- examples
- output requirements

In later chapters, this idea will grow into something much more important:

> The model only knows what we put into its context.

## 3. LLM calls are stateless

A single LLM call does not remember previous calls.

If we want the model to use earlier information, our application must send that information again.

#### Let's test a hypothesis

Here is a natural assumption:

> If I tell the model my name, it will remember it when I ask later.

Let's test that assumption.

🤔 **Think About It**

Before running the code, predict what will happen.

Will the second call know the name?

In [34]:
first_response = call_llm(
    "My name is Alice. Reply with exactly: Nice to meet you, Alice."
)

second_response = call_llm(
    "What is my name?"
)

print(Panel(first_response, title="First call"))
print(Panel(second_response, title="Second call"))

╭────────────────────────────────────────────────── First call ───────────────────────────────────────────────────╮
│ Nice to meet you, Alice.                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Second call ──────────────────────────────────────────────────╮
│ I don’t know your name. How can I assist you today?                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

#### What happened?


The second call usually does **not** know the name.

That is not because the model is broken.

It is because each API call is independent.

```mermaid
sequenceDiagram
    participant App as Python App
    participant LLM as LLM API

    App->>LLM: My name is Alice
    LLM-->>App: Nice to meet you, Alice

    App->>LLM: What is my name?
    LLM-->>App: I do not know
```

💡 **Key Idea**

A single LLM API call is **stateless**.

The model only sees the input you send in the current request.

## 4. Conversation history

The simplest way to give the model context is to include the previous conversation in the next request.

This works, but it quickly becomes messy if we treat the whole conversation as one long string.

#### Send the conversation ourselves

If the model only sees the current request, we can send the previous conversation again.

The simplest possible representation is one long string.

In [37]:
conversation_as_text = '''
User: My name is Alice.

Assistant: Nice to meet you, Alice.

User: What is my name?
'''

response = call_llm(conversation_as_text)

print(response)

Your name is Alice.

This works surprisingly well.

The model can read the transcript and infer that the user's name is Alice.

So we have discovered the basic chatbot trick:

> Keep the conversation somewhere, and send it back to the model.

#### But plain text history does not scale well


A transcript string is easy for a demo, but awkward for real applications.

Imagine a longer conversation:

```text
User: ...
Assistant: ...
User: ...
Assistant: ...
User: ...
Assistant: ...
```

Now ask:

- Where does one message end and another begin?
- How do we reliably tell user messages from assistant messages?
- How do we insert system instructions?
- How will this work when we later add tools?
- How will we inspect, edit or truncate history?

⚠️ **Common Mistake**

It is tempting to manage conversations as one giant string. That works for tiny demos, but becomes fragile as the application grows.

## 5. Structured messages

Instead of storing a conversation as one long string, we can store it as a list of messages.

Each message has a role and some content.

#### Messages: A better representation of conversation history


Instead of inventing our own transcript format, we use a structured list of messages.

Each **message** has:

- a `role`
- some `content`

This is the basic representation of a conversation.

```mermaid
flowchart TD
    A[Conversation] --> B[Message 1]
    A --> C[Message 2]
    A --> D[Message 3]

    B --> B1[role: user]
    B --> B2[content: My name is Alice]

    C --> C1[role: assistant]
    C --> C2[content: Nice to meet you]
```

This is why messages exist.

They give our application a clean way to represent conversation history.

#### Message roles

The most common roles are:

| Role | What it represents | Example |
|---|---|---|
| `system` | High-level behaviour and constraints | "You are a concise tutor." |
| `user` | Something the user said | "Explain APIs." |
| `assistant` | Something the model previously said | "An API is..." |

The role matters because the model treats the conversation as structured context, not just a blob of text.

✅ **Best Practice**

Use the `system` message for stable instructions.

Use `user` and `assistant` messages for the actual conversation.

Do not keep changing the system message every turn unless you have a good reason.

#### Rebuild the name example using messages


Now let's repeat the earlier experiment, but this time with a structured message list.

In [40]:
messages: list[dict[str, str]] = [
    {
        "role": "system",
        "content": ("You are a helpful assistant. Answer directly and do not invent information."),
    },
    {
        "role": "user",
        "content": "My name is Alice. Reply with exactly: Nice to meet you, Alice.",
    },
]

response = client.responses.create(
    model=MODEL,
    input=messages,
)

assistant_reply = response.output_text

print(assistant_reply)

Nice to meet you, Alice.

At this point, the model has responded.

But our Python programme still needs to store that response.

The model does not automatically save it for us.

In [41]:
messages.append(
    {
        "role": "assistant",
        "content": assistant_reply,
    }
)

messages.append(
    {
        "role": "user",
        "content": "What is my name?",
    }
)

response = client.responses.create(
    model=MODEL,
    input=messages,
)

print(response.output_text)

Your name is Alice.

#### Where did the memory come from?

The model did not remember Alice.

Our application remembered Alice by resending the conversation.

```mermaid
sequenceDiagram
    participant App as Python App
    participant History as Message History
    participant LLM as LLM API

    App->>History: Store user message
    App->>LLM: Send message history
    LLM-->>App: Assistant reply
    App->>History: Store assistant reply
    App->>LLM: Send updated history
    LLM-->>App: Follow-up answer
```

💡 **Key Idea**

The simplest chatbot memory is not inside the model.

It is the message history your application maintains and sends with each request.

#### Inspect the message history


Let's print the message list so we can see the context we sent.

In [42]:
from rich.table import Table

table = Table(title="Message History")
table.add_column("Index", style="bold")
table.add_column("Role")
table.add_column("Content", overflow="fold")

for index, message in enumerate(messages):
    table.add_row(
        str(index),
        message["role"],
        message["content"],
    )

print(table)

                                          Message History                                          
┏━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Index ┃ Role      ┃ Content                                                                     ┃
┡━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0     │ system    │ You are a helpful assistant. Answer directly and do not invent information. │
│ 1     │ user      │ My name is Alice. Reply with exactly: Nice to meet you, Alice.              │
│ 2     │ assistant │ Nice to meet you, Alice.                                                    │
│ 3     │ user      │ What is my name?                                                            │
└───────┴───────────┴─────────────────────────────────────────────────────────────────────────────┘

## 6. Building a simple chatbot

Now we can combine the pieces.

A chatbot is not a special kind of model. It is an application that keeps a message history, sends it to the model, stores the assistant response, and repeats.

```mermaid
flowchart LR
    A[Single LLM Call] --> B[Stateless]
    B --> C[Conversation History]
    C --> D[Structured Messages]
    D --> E[Chatbot]
```

#### Build it Manually first

Before writing a helper function, let's do the process manually.

Why?

Because if we jump straight to a function, it hides the important idea.

A chatbot turn has four steps:

1. Add the user's message to history.
2. Send the full history to the model.
3. Get the assistant's response.
4. Add the assistant's response to history.

```mermaid
flowchart TD
    A[Add user message] --> B[Send history to LLM]
    B --> C[Receive assistant response]
    C --> D[Add assistant response to history]
    D --> E[Ready for next turn]
```

🤔 **Think About It**

Where exactly is the chatbot's memory stored?

In [49]:
# An object to keep tract of chat histroy messages, starting with the system message.

chat_history: list[dict[str, str]] = [
    {
        "role": "system",
        "content": (
            "You are a concise AI tutor for intermediate Python developers. "
            "Use British English. "
            "keep your respone short."
        ),
    }
]

#### First manual turn

Let's add a user message, call the model, then store the assistant response.

In [50]:
# add the user question to the chat history
chat_history.append(
    {
        "role": "user",
        "content": "I'm learning about AI agents.",
    }
)

# get the LLMs response
response = client.responses.create(
    model=MODEL,
    input=chat_history,
)
assistant_response = response.output_text

# add the LLM response to the chat history
chat_history.append(
    {
        "role": "assistant",
        "content": assistant_response,
    }
)

print(assistant_response)

That’s great! AI agents can be fascinating. Are you focusing on a particular type, like reinforcement learning 
agents or rule-based agents?

#### Secondon manual turn

Now let's ask a follow-up question.

Notice that we do not only send the latest user message.

We send the entire message history.

In [52]:
# add the user question to the chat history
chat_history.append(
    {
        "role": "user",
        "content": "What am I learning about?",
    }
)

# get the LLMs response
response = client.responses.create(
    model=MODEL,
    input=chat_history,
)
assistant_response = response.output_text

# add the LLM response to the chat history
chat_history.append(
    {
        "role": "assistant",
        "content": assistant_response,
    }
)

print(assistant_response)

You’re learning about the principles and techniques behind AI agents, including their design, how they interact 
with environments, and the algorithms they use for decision-making. This could involve topics like machine 
learning, planning, and multi-agent systems. Let me know if you need clarification on a specific aspect!

#### Repetition is the point

Look at the code we just wrote twice.

Each turn repeats the same pattern:

```python
chat_history.append(user_message)

response = client.responses.create(...)

chat_history.append(assistant_response)
```

This repetition is useful.

It reveals the structure of a chatbot.

But we do not want to keep writing it manually.

💡 **Key Idea**

A chatbot is not a special kind of model. It is an application pattern built around repeatedly updating conversation history.

#### Let's refactor into a `chat()` function

Now that we understand the repeated pattern, we can turn it into a function.

This function will:

- accept the current history
- accept the latest user message
- call the model
- append both messages
- return the assistant's response

✅ **Best Practice**

Only introduce an abstraction after students have seen the repetition it removes.

In [54]:
def chat(
    history: list[dict[str, str]],
    user_message: str,
    model: str = MODEL,
) -> str:
    """Continue a conversation using message history.

    This function mutates `history` intentionally.

    The history list is the chatbot's short-term memory. Each call adds:
    1. the latest user message
    2. the assistant response

    Parameters
    ----------
    history:
        The existing conversation history.

    user_message:
        The latest message from the user.

    model:
        The model to call.

    Returns
    -------
    str
        The assistant's response.
    """

    history.append(
        {
            "role": "user",
            "content": user_message,
        }
    )

    response = client.responses.create(
        model=model,
        input=history,
    )
    assistant_message = response.output_text

    history.append(
        {
            "role": "assistant",
            "content": assistant_message,
        }
    )

    return assistant_message

#### Use the chatbot


Let's create a fresh conversation and use our `chat()` function.

In [59]:
course_chat: list[dict[str, str]] = [
    {
        "role": "system",
        "content": (
            "You are a helpful teaching assistant"
            "Your audience is intermediate Python developers. "
            "Keep your response short and practical. No more than a paragraph"
        ),
    }
]

In [60]:
# let's take it for a spin
reply = chat(
    course_chat,
    "I'm comfortable with Python functions but new to LLMs.",
)

print(reply)

Great! When working with Large Language Models (LLMs), think of them like advanced functions that generate text 
based on prompts. Instead of defining input-output pairs, you provide a prompt, and the model returns a response 
that often resembles coherent human-like text. Familiarize yourself with libraries like `transformers` from Hugging
Face, which allows you to easily load pre-trained models and interact with them using simple function calls. Start 
experimenting by crafting prompts and using `model.generate()` to see how the LLM responds. This hands-on practice 
will help you grasp their capabilities and nuances!

In [61]:
reply = chat(
    course_chat,
    "Based on my background, how should I think about LLM APIs?",
)

print(reply)

With your background in Python functions, approach LLM APIs like a powerful library that extends the capabilities 
of your code. Each API call can be thought of as invoking a complex function where you provide inputs (prompts) and
receive outputs (generated text). Focus on understanding the parameters you can pass, like temperature (for 
creativity) and max tokens (for length), to control the behavior of the model. Experiment with crafting different 
prompts to see how the model's outputs change, treating it like a sandbox for creative exploration and practical 
applications, such as chatbots or content generation!

#### Inspect the chatbot memory


Let's inspect the message history.

This is where the chatbot's memory lives.

In [62]:
from rich.table import Table

table = Table(title="Chatbot Message History")
table.add_column("Index", style="bold")
table.add_column("Role")
table.add_column("Content", overflow="fold")

for index, message in enumerate(course_chat):
    table.add_row(
        str(index),
        message["role"],
        message["content"],
    )

print(table)

                                              Chatbot Message History                                              
┏━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Index ┃ Role      ┃ Content                                                                                     ┃
┡━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0     │ system    │ You are a helpful teaching assistantYour audience is intermediate Python developers. Keep   │
│       │           │ your response short and practical. No more than a paragraph                                 │
│ 1     │ user      │ I'm comfortable with Python functions but new to LLMs.                                      │
│ 2     │ assistant │ Great! When working with Large Language Models (LLMs), think of them like advanced          │
│       │           │ functions that generate text based on prompts. Instead of defining input-output pairs, you  │
│       │           │ provide a prompt, and the model returns a response that often resembles coherent human-like │
│       │           │ text. Familiarize yourself with libraries like `transformers` from Hugging Face, which      │
│       │           │ allows you to easily load pre-trained models and interact with them using simple function   │
│       │           │ calls. Start experimenting by crafting prompts and using `model.generate()` to see how the  │
│       │           │ LLM responds. This hands-on practice will help you grasp their capabilities and nuances!    │
│ 3     │ user      │ Based on my background, how should I think about LLM APIs?                                  │
│ 4     │ assistant │ With your background in Python functions, approach LLM APIs like a powerful library that    │
│       │           │ extends the capabilities of your code. Each API call can be thought of as invoking a        │
│       │           │ complex function where you provide inputs (prompts) and receive outputs (generated text).   │
│       │           │ Focus on understanding the parameters you can pass, like temperature (for creativity) and   │
│       │           │ max tokens (for length), to control the behavior of the model. Experiment with crafting     │
│       │           │ different prompts to see how the model's outputs change, treating it like a sandbox for     │
│       │           │ creative exploration and practical applications, such as chatbots or content generation!    │
└───────┴───────────┴─────────────────────────────────────────────────────────────────────────────────────────────┘

#### What did we just build?

We built this:

```mermaid
flowchart LR
    U[User message] --> H[Append to history]
    H --> L[Send full history to LLM]
    L --> R[Assistant response]
    R --> S[Append response to history]
    S --> H
```

The loop is simple, but powerful.

The model appears to remember because our application keeps resending the conversation.

⚠️ **Common Mistake**

Do not say "the model remembered".

More accurate:

> The application maintained context and sent it back to the model.

## 7. Short-term memory has limits

Conversation history is useful, but it is not free.

As the conversation grows:

- the request becomes larger
- cost can increase
- latency can increase
- the model may receive irrelevant old context
- the conversation may eventually exceed the context window

We will revisit this later when we discuss memory, state and context management.

For now, the important lesson is:

> Basic chatbot memory is just message history.

## 8. Chapter takeaway

We started with the smallest useful AI application: one LLM call.

Then we discovered a problem:

> The model does not remember previous calls.

We tried sending the conversation as plain text.

That worked, but it was fragile.

So we introduced structured messages.

Finally, we turned the repeated pattern into a `chat()` function.

```mermaid
flowchart LR
    A[LLM Call] --> B[Stateless]
    B --> C[Need History]
    C --> D[Messages]
    D --> E[chat function]
    E --> F[Chatbot]
```

💡 **Key Idea**

A chatbot is an LLM plus conversation history.

The memory is not magic.

The memory is the context your application sends to the model.

---
---

# Chapter 2 — From Plain Text to Structured Data

## Learning objectives

By the end of this chapter, you will be able to:

- Explain why free-form text is difficult for software to use reliably.
- Recognise the limitations of manually parsing model output.
- Define a schema for model output.
- Use Structured Outputs to get typed data from an LLM.
- Explain why structured data is an important step towards agents.

## 1. Why text is hard for software

Imagine we're building an AI assistant.

A user says:

> *I'm travelling to Tokyo next Tuesday with my wife.*

As humans, we immediately understand:

- Destination: Tokyo
- Departure: Next Tuesday
- Travellers: 2

But our Python code doesn't.

```mermaid
flowchart LR
User-->LLM
LLM-->Paragraph
Paragraph-->Python
Python-->Q{Can I reliably
understand this?}
```

🤔 **Think About It**

If you were writing this application without any new OpenAI features, how would you extract those three pieces of information?

In [70]:
travel_text = call_llm(
    "Extract the travel details from this request:"
    "'I'm travelling to Tokyo next Tuesday with my wife.'"
)

print(travel_text)

Travel Details:
- Destination: Tokyo
- Date: Next Tuesday
- Travellers: You and your wife

The response looks perfectly reasonable.

A human can read it immediately. Unfortunately, our application still has work to do.

It must somehow find:

- destination
- departure date
- number of travellers

inside a paragraph.

## 2. The fragile approach: parsing text

A common instinct is to parse the response ourselves. For simple examples, regular expressions appear to work.

In [71]:
import re

destination = re.search(r"Tokyo|Paris|London", travel_text)

print(destination.group() if destination else "Not found")

Tokyo

For this example, the regex succeeds.

So are we done? Not quite.

Let's imagine the model responds differently.

Instead of:

> Destination: Tokyo

it says

> You'll be heading to Japan's capital city next Tuesday.

Or perhaps:

> Your trip is planned for Tokyo.

Or maybe it changes the formatting completely.

Our parsing logic quickly becomes fragile.

In [72]:
alternative = '''
Your trip is planned for Tokyo and you'll be travelling next Tuesday with one companion.
'''

print(alternative)

Your trip is planned for Tokyo and you'll be travelling next Tuesday with one companion.

Now ask yourself: Would our regex still work?

Maybe.

Would it continue working after hundreds of different phrasings?

Probably not.

The problem isn't regular expressions.

The problem is that **paragraphs are designed for people, not software.**

💡 **Key Idea**

Humans enjoy reading prose.

Software prefers structured data.

Instead of teaching Python how to understand every possible paragraph...

...what if we asked the model to return structured data in the first place?

## 3. Schemas describe the shape of data

#### Introducing Schemas


In the previous section, we discovered the core problem:

> Free-form text is useful for humans, but awkward for software.

Our regex example worked only because the example was tiny.

A better approach is to describe the **shape of the data** we want.

That shape is called a **schema**.

```mermaid
flowchart LR
    A[User request] --> B[LLM]
    B --> C[Schema]
    C --> D[Validated Python object]
```

💡 **Key Idea**

A schema is a contract between your application and the model.

It tells the model:

> "Do not just write a nice paragraph. Return data in this exact shape."

#### Introducing Pydantic

In Python, one of the most common ways to define schemas is with **Pydantic**.

Pydantic lets us define:

- field names
- field types
- descriptions
- validation rules

This is useful because our application can then work with normal Python objects rather than scraping values from text.

✅ **Best Practice**

Use meaningful field names and short descriptions. The model uses both when deciding how to fill the object.

#### Creating a schema using Pydantic

Let's create a schema for the travel request example discussed earlier.

The user said:

> *I'm travelling to Tokyo next Tuesday with my wife.*

We want to extract:

- destination
- departure date
- number of travellers

In [80]:
from pydantic import BaseModel, Field

class TravelRequest(BaseModel):
    """Structured representation of a user's travel request."""

    destination: str = Field(description="The city or location the user wants to travel to.")
    departure_date: str = Field(description="The user's intended departure date, as stated or inferred.")
    travellers: int = Field(description="The total number of people travelling, including the user.")

Notice what changed.

We are no longer asking the model for "a helpful answer".

We are telling it the exact structure our software needs.

```mermaid
flowchart LR
    A[Paragraph] -. fragile .-> B[Regex]
    C[Schema] --> D[Python object]
```

## 5. Structured Outputs


Structured Outputs let us ask the model for data that matches a schema.

Instead of receiving a paragraph and trying to parse it ourselves, we receive a typed object.

#### Using Schema with LLM calls to return structed outputs

Now we can ask the model to return the result as a `TravelRequest`.

The Responses API gives us a convenient method for this: `client.responses.parse()`.

Instead of returning only text, it can return an object that matches our Pydantic schema.

In [82]:
travel_request = client.responses.parse(
    model=MODEL,
    input=(
        "Extract the travel details from this request:\n"
        "'I'm travelling to Tokyo next Tuesday with my wife.'"
    ),
    text_format=TravelRequest,
).output_parsed

travel_request

TravelRequest(destination='Tokyo', departure_date='next Tuesday', travellers=2)

#### Now we can use the LLM response as part of normal python logic

The result is now a Python object.

That means our application can use fields directly.

In [85]:
print(f"Destination: {travel_request.destination}")
print(f"Departure date: {travel_request.departure_date}")
print(f"Travellers: {travel_request.travellers}")

Destination: Tokyo

Departure date: next Tuesday

Travellers: 2

#### Let's look at another Example


Let's prove that this idea is not specific to travel.

Suppose our assistant receives meeting requests.

The user might say:

> *Schedule a project review with Maya tomorrow at 3pm for 45 minutes.*

Our application needs structured data, not a paragraph.

In [86]:
class MeetingRequest(BaseModel):
    """Structured representation of a meeting request."""

    title: str = Field(description="Short title for the meeting.")
    attendee: str = Field(description="The person the user wants to meet.")
    date: str = Field(description="The requested meeting date.")
    time: str = Field(description="The requested meeting time.")
    duration_minutes: int = Field(description="Meeting duration in minutes.")

In [87]:
meeting_request = client.responses.parse(
    model=MODEL,
    input=(
        "Extract the meeting details from this request:\n"
        "'Schedule a project review with Maya tomorrow at 3pm for 45 minutes.'"
    ),
    text_format=MeetingRequest,
).output_parsed

meeting_request

MeetingRequest(title='Project Review', attendee='Maya', date='2023-10-06', time='15:00', duration_minutes=45)

In [88]:
print(f"Title: {meeting_request.title}")
print(f"Attendee: {meeting_request.attendee}")
print(f"Date: {meeting_request.date}")
print(f"Time: {meeting_request.time}")
print(f"Duration: {meeting_request.duration_minutes} minutes")

Title: Project Review

Attendee: Maya

Date: 2023-10-06

Time: 15:00

Duration: 45 minutes

The pattern is the same.

Only the schema changed.

```mermaid
flowchart TD
    A[User request] --> B[Model]
    B --> C{Which schema?}
    C --> D[TravelRequest]
    C --> E[MeetingRequest]
    D --> F[Python object]
    E --> F
```

📌 **Checkpoint**

We now have a reliable bridge between language and software.

The model can still understand natural language, but our Python code receives structured data.

## 6. Where Structured Outputs Fit in Agentic Systems

Structured outputs are one of the foundations that make reliable agents possible.

Why?

Because before an agent can take useful action, it often needs to understand the user's request as data.

```mermaid
flowchart LR
    U[User request] --> L[LLM]
    L --> S[Structured data]
    S --> A[Application decision]
    A --> T[Action]
```

For example:

| User says | Structured data enables |
|---|---|
| "Book a table for four" | reservation workflow |
| "Schedule a meeting tomorrow" | calendar workflow |
| "Summarise this invoice" | accounting workflow |
| "Find flights under £500" | search and filtering workflow |

The model understands language.

Your application needs data.

Structured outputs connect the two.

## 7. Chapter Summary

In Chapter 1, we built a chatbot.

In Chapter 2, we solved a new problem:

> Text responses are useful for humans, but difficult for software to consume.

We tried parsing text.

That was fragile.

Then we introduced schemas and structured outputs.

```mermaid
flowchart LR
    A[Free text] --> B[Parsing problem]
    B --> C[Schema]
    C --> D[Structured output]
    D --> E[Python object]
```

💡 **Key Idea**

Structured outputs create a contract between your application and the model.

Instead of scraping values from prose, your application receives validated data.

---
---

# Chapter 3 — From Knowledge to Action

## Learning objectives

By the end of this chapter, you will be able to:

- Explain why knowledge alone is not enough for an agent.
- Write Python functions that can be used as tools.
- Describe tools so a model can request them.
- Use native tool calling with the OpenAI Responses API.
- Explain the difference between the model requesting a tool and Python executing it.

---

So far, our application can hold a conversation and return structured data.

But it still cannot do anything inside our application.

To move from useful text to useful action, we need tools.

```mermaid
flowchart LR
    A[User request] --> B[Model]
    B --> C[Tool request]
    C --> D[Python function]
    D --> E[Tool result]
    E --> F[Final answer]
```

The important idea is simple:

> The model chooses an action. Your application executes it.

## 1. Knowledge is not action


An LLM can generate useful text, but it cannot directly perform actions inside your application.

For example, it can explain what a weather function might do.

But it cannot run your weather function unless your application exposes that function as a tool.

| Capability | Example |
|---|---|
| Knowledge | “Weather reports usually include temperature and conditions.” |
| Action | Run `get_weather("Almaty")` and return the result. |

This distinction is central to agents.

Agents are not just systems that know things.

They are systems that can use tools to do things.

## 2. Tools are Python functions


A tool is just a Python function that the application is willing to let the model request.

Before a function becomes an AI tool, it is ordinary Python.

Let's start without the model.

In [9]:
def get_weather(city: str) -> str:
    """Return a short weather report for a city."""

    weather_by_city = {
        "london": "Cloudy, 14°C, light rain expected later.",
        "paris": "Sunny, 22°C, gentle breeze.",
        "almaty": "Warm, 27°C, humid.",
        "astana": "Clear, 21°C, chilly.",
    }

    normalised_city = city.strip().lower()

    return weather_by_city.get(
        normalised_city,
        f"No weather data available for {city}.",
    )

In [10]:
get_weather("Almaty")

'Warm, 27°C, humid.'

We can also expose other simple functions.

A calculator is a useful teaching example because the model may know how to reason about arithmetic, but Python can execute the calculation exactly.

In [11]:
def calculate(expression: str) -> str:
    """Evaluate a simple arithmetic expression."""

    allowed_characters = set("0123456789+-*/(). ")

    if not set(expression) <= allowed_characters:
        return "Error: expression contains unsupported characters."

    try:
        result = eval(expression, {"__builtins__": {}}, {})
    except Exception as exc:
        return f"Error: {exc}"

    return str(result)

In [12]:
calculate("17 * 23")

'391'

⚠️ **Common Mistake**

The model does not automatically know that these Python functions exist.

To make them usable, our application must describe them to the model.

## 3. Describing tools to the model

The model cannot inspect our Python code directly.

If we want it to request a function, we need to describe that function in a format the model can understand.

A tool description usually includes:

- the tool name
- what the tool does
- what arguments it expects

```mermaid
flowchart LR
    A[Python function] --> B[Tool description]
    B --> C[Model can request tool]
```

💡 **Key Idea**

A tool schema is not the tool itself.

It is a description of a Python function that the model is allowed to request.

In [13]:
weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Return a short weather report for a city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city to get the weather for.",
            }
        },
        "required": ["city"],
        "additionalProperties": False,
    },
}

In [14]:
calculator_tool = {
    "type": "function",
    "name": "calculate",
    "description": "Evaluate a simple arithmetic expression.",
    "parameters": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "A simple arithmetic expression, such as '17 * 23'.",
            }
        },
        "required": ["expression"],
        "additionalProperties": False,
    },
}

In [15]:
tools = [weather_tool, calculator_tool]

We now have two things:

1. the real Python functions
2. descriptions of those functions for the model

The next step is to let the model decide whether it wants to request one of them.

## 4. Native tool calling

Now we pass the tool descriptions to the model.

The model can either answer normally or request a tool call.

At this stage, we are not executing anything yet.

We are only asking the model what it wants to do.

In [16]:
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"

In [17]:
response = client.responses.create(
    model=MODEL,
    input="What is the weather in Almaty?",
    tools=tools,
)

response.output

[ResponseFunctionToolCall(arguments='{"city":"Almaty"}', call_id='call_Rg8cK1qe1YRaLF0YYEDfU8Nu', name='get_weather', type='function_call', id='fc_0d5c96e5e79b7c22006a44d4182370819f963a212b022eb116', namespace=None, status='completed')]

The response may contain a tool call.

That means the model has decided that the user request is better answered by using one of the tools we provided.

At this point, the model has not checked the weather.

It has only requested that our application call `get_weather`.

In [18]:
tool_call = next(
    item for item in response.output
    if item.type == "function_call"
)

tool_call

ResponseFunctionToolCall(arguments='{"city":"Almaty"}', call_id='call_Rg8cK1qe1YRaLF0YYEDfU8Nu', name='get_weather', type='function_call', id='fc_0d5c96e5e79b7c22006a44d4182370819f963a212b022eb116', namespace=None, status='completed')

In [19]:
tool_call.name, tool_call.arguments

('get_weather', '{"city":"Almaty"}')

## 5. Python executes the tool

The model has requested a tool call.

But the model has not executed Python.

Our application must now:

1. read the requested tool name
2. read the arguments
3. call the matching Python function
4. capture the result

In [20]:
import json

arguments = json.loads(tool_call.arguments)

arguments

{'city': 'Almaty'}

In [21]:
tool_result = get_weather(**arguments)

tool_result

'Warm, 27°C, humid.'

For one tool, manual execution is fine.

But once we have several tools, we want one small dispatcher function.

A dispatcher reads the tool name and routes the request to the correct Python function.

In [22]:
def run_tool(tool_call) -> str:
    """Execute a tool call requested by the model."""

    tool_name = tool_call.name
    arguments = json.loads(tool_call.arguments)

    if tool_name == "get_weather":
        return get_weather(**arguments)

    if tool_name == "calculate":
        return calculate(**arguments)

    raise ValueError(f"Unknown tool: {tool_name}")

In [23]:
tool_result = run_tool(tool_call)

tool_result

'Warm, 27°C, humid.'

This is the key boundary in tool calling:

| Step | Who does it? |
|---|---|
| Choose the tool | Model |
| Execute the tool | Python application |
| Return the result | Python application |
| Write the final answer | Model |

The model requests.

The application executes.

## 6. Returning tool results to the model

After Python runs the tool, the model still needs to see the result.

So the application sends the tool result back to the model.

The model can then produce a normal final answer for the user.

In [24]:
final_response = client.responses.create(
    model=MODEL,
    previous_response_id=response.id,
    input=[
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": tool_result,
        }
    ],
)

print(final_response.output_text)

The weather in Almaty is warm, with a temperature of 27°C and humid conditions.

We have now completed one full tool-use cycle.

```mermaid
sequenceDiagram
    participant User
    participant Model
    participant Python

    User->>Model: What is the weather in Almaty?
    Model->>Python: Call get_weather(city="Almaty")
    Python->>Python: Execute get_weather("Almaty")
    Python->>Model: Tool result
    Model->>User: Final answer
```

This is not yet a full agent loop.

It is one complete tool call from start to finish.

## Optional short exercise


Add one more city to the `get_weather()` function.

Then ask the model about that city and run the same tool-calling flow again.

Suggested city:

```python
"berlin": "Cool, 12°C, cloudy with a chance of rain."
```

Keep the exercise short. The goal is to practise the flow, not build a production weather system.

## 7. Chapter takeaway

In this chapter, we moved from knowledge to action.

A model does not execute tools directly.

It requests a tool call, and your Python application executes it.

```mermaid
flowchart LR
    A[User request] --> B[Model]
    B --> C[Tool call]
    C --> D[Python executes function]
    D --> E[Tool result returned to model]
    E --> F[Model writes final answer]
```

We have now completed one tool-use cycle:

1. describe tools to the model
2. let the model request a tool
3. execute the tool in Python
4. return the result to the model
5. receive the final answer

The limitation is that we handled the cycle manually.

In the next chapter, we will turn this repeated process into an agent loop.

---
---

---
---

# Chapter 4 — Building an Agent

## Learning objectives

By the end of this chapter, you will be able to:

- Explain why one-shot tool calling is not enough for multi-step tasks.
- Describe the agent loop: think, act, observe, repeat.
- Build a small `run_agent()` function from scratch.
- Explain how context changes during an agent run.
- Use stopping conditions to keep an agent loop bounded.

---

In Chapter 3, we completed one tool-use cycle manually.

The model requested a tool.  
Python executed it.  
The result went back to the model.  
The model produced a final answer.

That was useful, but it was still manual.

Real tasks often require the same pattern to repeat.

```mermaid
flowchart LR
    A[User request] --> B[Model thinks]
    B --> C{Needs a tool?}
    C -- Yes --> D[Python executes tool]
    D --> E[Tool result]
    E --> B
    C -- No --> F[Final answer]
```

That repeating pattern is the agent loop.

## 1. The limitation of one-shot tool calling


In Chapter 3, we handled one tool call by hand.

That worked because the task was small:

> What is the weather in Almaty?

But many useful tasks are more open-ended:

> What is the weather in Almaty, and what is 17 multiplied by 23?

The model may need to:

1. request the weather tool
2. observe the result
3. request the calculator tool
4. observe the result
5. produce a final answer

Writing those steps manually every time would not scale.

We need a loop.

## 2. The agent loop

The core agent loop is simple:

```text
while True:
    ask the model what to do next

    if the model gives a final answer:
        stop

    if the model requests a tool:
        run the tool
        send the result back to the model
```

This is often described as:

```mermaid
flowchart LR
    A[Think] --> B[Act]
    B --> C[Observe]
    C --> A
```

In our implementation:

| Agent step | What happens |
|---|---|
| Think | The model reads the current context |
| Act | The model requests a tool call |
| Observe | Python returns the tool result |
| Repeat | The model sees the updated context |

💡 **Key Idea**

An agent is not a new kind of model.

It is an application pattern around a model: context, tools, a loop and stopping conditions.

## 3. Building `run_agent()`

We will now build a small agent loop from scratch.

The agent loop needs a small helper to find tool calls in a response.

A model response may contain different output items.

For this chapter, we only care about items whose type is `function_call`.

In [25]:
def collect_tool_calls(response) -> list:
    """Return all tool calls from a model response."""

    return [
        item
        for item in response.output
        if item.type == "function_call"
    ]

Now we can build the first version of `run_agent()`.

The function will:

1. send the user's task to the model
2. check whether the model requested tools
3. execute each requested tool
4. send the tool results back
5. repeat until the model gives a final answer

In [26]:
def run_agent(task: str, max_iterations: int = 5) -> str:
    """Run a small tool-using agent loop."""

    response = client.responses.create(
        model=MODEL,
        input=task,
        tools=tools,
    )

    for _ in range(max_iterations):
        tool_calls = collect_tool_calls(response)

        if not tool_calls:
            return response.output_text

        tool_outputs = []

        for tool_call in tool_calls:
            tool_result = run_tool(tool_call)

            tool_outputs.append(
                {
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": tool_result,
                }
            )

        response = client.responses.create(
            model=MODEL,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools,
        )

    return "The agent stopped because it reached the maximum number of iterations."

Try the agent with a task that may require more than one tool.

In [27]:
answer = run_agent(
    "What is the weather in Almaty, and what is 17 multiplied by 23?"
)

print(answer)

The weather in Almaty is warm, at 27°C with high humidity. Additionally, 17 multiplied by 23 equals 391.

We have now written a simple agent.

The important part is not the amount of code.

The important part is the pattern:

```mermaid
flowchart TD
    A[Start with user task] --> B[Call model]
    B --> C{Tool calls?}
    C -- No --> D[Return final answer]
    C -- Yes --> E[Run tools in Python]
    E --> F[Send tool outputs back]
    F --> B
```

## 4. Making the loop visible (Traceability)


When teaching or debugging agents, it helps to see the loop.

Let's create a slightly more verbose version that prints what is happening.

In [30]:
def run_agent_verbose(task: str, max_iterations: int = 5) -> str:
    """Run the agent loop and print each step."""

    print("User task:")
    print(task)
    print()

    response = client.responses.create(
        model=MODEL,
        input=task,
        tools=tools,
    )

    for iteration in range(1, max_iterations + 1):
        print(f"Iteration {iteration}")

        tool_calls = collect_tool_calls(response)

        if not tool_calls:
            print("No tool calls requested.")
            print()
            return response.output_text

        tool_outputs = []

        for tool_call in tool_calls:
            print(f"Tool requested: {tool_call.name}")
            print(f"Arguments: {tool_call.arguments}")

            tool_result = run_tool(tool_call)

            print(f"Tool result: {tool_result}")
            print()

            tool_outputs.append(
                {
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": tool_result,
                }
            )

        response = client.responses.create(
            model=MODEL,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools,
        )

    return "The agent stopped because it reached the maximum number of iterations."

In [29]:
answer = run_agent_verbose(
    "What is the weather in London, and what is 45 divided by 9?"
)

print("Final answer:")
print(answer)

User task:

What is the weather in London, and what is 45 divided by 9?

Iteration 1

Tool requested: get_weather

Arguments: {"city":"London"}

...

Tool result: 5.0

Iteration 3

No tool calls requested.

Final answer:

The weather in London is currently cloudy with a temperature of 14°C, and light rain is expected later. 

Also, 45 divided by 9 is 5.

This version is not more agentic.

It is simply easier to inspect.

For production systems, you would usually use proper logging or framework-level tracing.

For teaching, printing the loop is enough.

## 5. Agent Context


We now have a working agent loop.

But there is one idea we need to make explicit:

> Every time the loop calls the model, what information does the model receive?

That information is the **agent context**.

```mermaid
flowchart TD
    A[Instructions] --> C[Agent Context]
    B[User Request] --> C
    D[Available Tools] --> C
    E[Tool Results So Far] --> C
    F[Conversation History] --> C
    C --> G[LLM Call]
```

💡 **Key Idea**

An agent does not remember by itself.

Your application remembers by rebuilding and sending context on each model call.

#### What is inside the context?

For our small scratch agent, the context includes:

| Context piece | Where it appears |
|---|---|
| User request | Initial `input` |
| Available tools | `tools=tools` |
| Tool results | `function_call_output` messages |
| Previous model response | `previous_response_id` |
| Stopping rule | `max_iterations` in our loop |

In larger systems, context may also include:

- conversation history
- retrieved documents
- user preferences
- application state
- safety instructions
- previous decisions

The model only sees what we include.

####  Context changes over time

The important part is that context is not static.

Each time the agent acts, the context becomes richer.

```mermaid
sequenceDiagram
    participant Agent
    participant Model
    participant Tool

    Agent->>Model: User request + tools
    Model-->>Agent: Tool request
    Agent->>Tool: Execute tool
    Tool-->>Agent: Tool result
    Agent->>Model: Previous response + tool result
    Model-->>Agent: Final answer or another tool request
```

This is why the loop works.

The model can make a better decision on the second call because it now has the observation from the first call.

## 6. Stopping conditions

Every agent loop needs a stopping condition.

Without one, a bug or confusing task could keep the loop running for too long.

Our implementation stops when either:

1. the model produces a final answer with no tool calls
2. the loop reaches `max_iterations`

```python
for _ in range(max_iterations):
    ...
```

For a teaching agent, this is enough.

For a production agent, stopping conditions become more important because tools may cost money, take time, or change real systems.

In [31]:
run_agent(
    "Tell me the weather in Paris.",
    max_iterations=3,
)

'The weather in Paris is sunny with a temperature of 22°C and a gentle breeze.'

## Optional short exercise


Change the `max_iterations` value and run the agent again.

Try:

```python
run_agent(
    "What is the weather in New York, and what is 100 / 4?",
    max_iterations=1,
)
```

What happens if the agent needs more steps than you allow?

This is a small exercise, but it teaches an important lesson: agent loops must be bounded.

## 7. Chapter takeaway

---
---

# Chapter 5 — From Scratch to the OpenAI Agents SDK


## Learning objectives


By the end of this chapter, you will be able to:

- Identify the agent plumbing we built manually.
- Explain why the Agents SDK is useful after building an agent from scratch.
- Create and run a simple SDK agent.
- Add Python tools using `@function_tool`.
- Compare the scratch implementation with the SDK version.

---

We have now built a small agent from scratch.

That was valuable because it showed the mechanics:

- model calls
- tool descriptions
- tool execution
- tool results
- context
- looping
- stopping conditions

But most of that code is plumbing.

The OpenAI Agents SDK packages much of that plumbing into cleaner primitives.

```mermaid
flowchart LR
    A[Scratch agent] --> B[Understand the mechanics]
    B --> C[Agents SDK]
    C --> D[Build with less plumbing]
```

The SDK does not make agents possible.

We already built one.

The SDK makes agents easier to build and maintain.

## 1. What did we build ourselves?

In Chapter 4, our scratch agent handled several responsibilities manually.

| Concern | Scratch implementation |
|---|---|
| Model call | `client.responses.create(...)` |
| Tool descriptions | manual tool schemas |
| Tool execution | `run_tool(...)` |
| Tool loop | `for` loop inside `run_agent()` |
| Context continuation | `previous_response_id=response.id` |
| Stopping condition | `max_iterations` |
| Visibility | `print(...)` statements |

This was the right way to learn.

But in a larger application, we do not want to keep rewriting the same loop and tool-dispatch code.

We want to focus on the behaviour of the agent and the tools it can use.

## 2. Why the SDK exists

The SDK gives us higher-level building blocks for the same ideas.

The two most important objects for this chapter are:

| SDK concept | Purpose |
|---|---|
| `Agent` | Defines the agent's name, instructions, model and tools |
| `Runner` | Runs the agent loop |

The mental model is:

```mermaid
flowchart LR
    A[Agent] --> B[Runner]
    B --> C[Model calls]
    B --> D[Tool execution]
    B --> E[Looping]
    B --> F[Final result]
```

Instead of writing the loop ourselves, we describe the agent and let the runner execute it.

## 3. Your first SDK agent


Install the SDK if needed:

```bash
pip install openai-agents
```

Make sure your OpenAI API key is available in your environment.

In notebooks, we can run SDK agents with:

```python
result = await Runner.run(agent, "Your task here")
```

The result contains a `final_output`.

In [36]:
from agents import Agent, Runner

MODEL = "gpt-4o-mini"

An SDK agent has:

- a name
- instructions
- optionally a model
- optionally tools



Let's create the simplest possible SDK agent.

This one has instructions, but no tools yet.

In [38]:
teaching_assistant = Agent(
    name="Teaching Assistant",
    instructions=(
        "You are a concise teaching assistant for a course on AI agents. "
        "Your audience is intermediate Python developers. "
        "Keep your responses short."
    ),
    model=MODEL,
)

In [39]:
result = await Runner.run(
    teaching_assistant,
    "Explain what an agent is in two sentences.",
)

print(result.final_output)

An agent is an entity that perceives its environment through sensors and acts upon that environment using 
effectors, making decisions based on its goals. In AI, agents often utilize algorithms to learn and adapt their 
behavior to achieve specific tasks effectively.

This is already much cleaner than manually calling the model.

But the real payoff appears when we add tools.

## 4. Adding tools with `@function_tool`


In the scratch version, we had to write a Python function and a separate tool schema.

With the SDK, we can wrap a Python function with `@function_tool`.

The function remains ordinary Python, but the SDK can expose it to the agent as a tool.


```mermaid
flowchart LR
    A[Python function] --> B["@function_tool"]
    B --> C[Agent tool]
    C --> D[Runner executes it when requested]
```

In [40]:
from agents import function_tool

In [48]:
@function_tool
def get_weather(city: str) -> str:
    """Return a short weather report for a city."""

    weather_by_city = {
        "london": "Cloudy, 14°C, light rain expected later.",
        "paris": "Sunny, 22°C, gentle breeze.",
        "almaty": "Warm, 27°C, humid.",
        "astana": "Clear, 21°C, chilly.",
    }

    normalised_city = city.strip().lower()

    return weather_by_city.get(
        normalised_city,
        f"No weather data available for {city}.",
    )

In [49]:

def _calculate(expression: str) -> str:
    """Evaluate a simple arithmetic expression."""

    allowed_characters = set("0123456789+-*/(). ")

    if not set(expression) <= allowed_characters:
        return "Error: expression contains unsupported characters."

    try:
        result = eval(expression, {"__builtins__": {}}, {})
    except Exception as exc:
        return f"Error: {exc}"

    return str(result)

calculate = function_tool(_calculate)

That is much less code than the scratch version.

We did not manually write:

- JSON tool schemas
- argument parsing
- a dispatcher
- function-call output messages

The SDK handles those details.

In [50]:
# The SDK sends all this info on the tool to the Model
print(f" Tool Name: {calculate.name}")
print(f"Description: {calculate.description}")
print(f"Parameters: {calculate.params_json_schema}")
print(f" Should be strict JSON:{calculate.strict_json_schema}")

Tool Name: _calculate

Description: Evaluate a simple arithmetic expression.

Parameters: {'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 
'title': '_calculate_args', 'type': 'object', 'additionalProperties': False}

Should be strict JSON:True

## 5. Rebuilding the agent with the SDK


Now we can create an agent that has access to the same tools.

The agent definition contains the behaviour.

The runner handles the execution.

In [51]:
practical_assistant = Agent(
    name="Practical Assistant",
    instructions=(
        "You are a practical assistant. "
        "Use tools when they help answer the user's question. "
        "Answer clearly and concisely in British English."
    ),
    model=MODEL,
    tools=[
        get_weather,
        calculate,
    ],
)

In [53]:
result = await Runner.run(
    practical_assistant,
    "What is the weather in Almaty, and what is 17 multiplied by 23?",
)

print(result.final_output)

The weather in Almaty is warm at 27°C and humid. Additionally, 17 multiplied by 23 equals 391.

This is the same kind of task we handled in Chapter 4.

The difference is that we no longer wrote the loop ourselves.

The SDK runner can:

1. call the model
2. handle tool calls
3. execute SDK tools
4. send tool results back
5. continue until there is a final answer

That is the same pattern we built from scratch.

## 6. Scratch vs SDK


Here is the most important comparison.

| Scratch implementation | Agents SDK |
|---|---|
| `client.responses.create(...)` | `Runner.run(...)` |
| Manual tool schemas | `@function_tool` |
| Manual argument parsing | SDK handles arguments |
| Manual `run_tool(...)` dispatcher | SDK executes matching tools |
| Manual tool-result messages | SDK manages tool results |
| Manual loop | Runner orchestration |
| Manual context continuation | SDK-managed run |
| Manual stopping logic | SDK run controls |

💡 **Key Idea**

The SDK does not replace the concepts.

It packages the concepts into cleaner primitives.

You should now be able to look at this short SDK example and understand what is happening underneath.

```mermaid
flowchart TD
    A[User task] --> B[Runner.run]
    B --> C[Model decides]
    C --> D{Tool needed?}
    D -- Yes --> E[SDK executes tool]
    E --> F[Tool result]
    F --> C
    D -- No --> G[Final output]
```

The SDK hides the plumbing, but the underlying loop should no longer feel mysterious.

## Optional short exercise

Add one new SDK tool.

For example:

```python
@function_tool
def convert_gbp_to_usd(amount_gbp: float) -> str:
    """Convert GBP to USD using a fixed teaching exchange rate."""

    exchange_rate = 1.25
    amount_usd = amount_gbp * exchange_rate
    return f"£{amount_gbp:.2f} is approximately ${amount_usd:.2f}"
```

Then add it to an agent and ask:

```python
result = await Runner.run(
    currency_agent,
    "Roughly how many dollars is £80?",
)
```

Keep this short.

The goal is to practise the SDK pattern, not build a production finance tool.

## 7. Day 1 takeaway

Today we built agents from first principles.

We started with one LLM call and gradually added the pieces needed for agentic behaviour.

```mermaid
flowchart LR
    A[LLM Call] --> B[Chatbot]
    B --> C[Structured Outputs]
    C --> D[Tools]
    D --> E[Agent Loop]
    E --> F[Agents SDK]
```

The full journey was:

| Chapter | Main idea |
|---|---|
| Chapter 1 | A chatbot is an application that manages conversation history |
| Chapter 2 | Structured outputs make model responses usable by software |
| Chapter 3 | Tools let the model request actions through Python |
| Chapter 4 | An agent loop lets the model act, observe and repeat |
| Chapter 5 | The SDK packages the same pattern with less plumbing |

The most important lesson is:

> The SDK does not make agents possible. It makes them practical.

Because you built the loop from scratch first, the SDK should now feel like a useful abstraction rather than a black box.

---
---

# Lab Brief


Use the ideas below to practise and extend what we covered today.

You do not need to complete every task. Choose one or two areas to explore, depending on your confidence and interests.

### 1. Modify the chatbot

- Change the system message.
- Try different personalities or roles.
- Add a few conversation turns.
- Inspect the message history.

### 2. Create a new structured output

- Define a new Pydantic model.
- Extract structured data from a messy user request.
- Use the parsed fields in simple Python logic.

### 3. Add a new tool

- Write a simple Python function.
- Describe it as a tool.
- Let the model request it.
- Execute it and return the result.

### 4. Extend the scratch agent

- Add your new tool to the existing agent.
- Update the tool dispatcher.
- Run the agent on a task that needs more than one tool.

### 5. Build an SDK agent

- Convert one or two tools using `@function_tool`.
- Create an `Agent`.
- Run it with `Runner.run()`.

### 6. Create a themed mini-agent

Choose one theme and build a small agent around it:

- study assistant
- travel assistant
- course support assistant
- finance helper
- productivity assistant

### 7. Compare scratch vs SDK

Reflect on the two implementations:

- What code did you write manually in the scratch version?
- What did the SDK remove?
- What still needs to be designed carefully?

### Suggested goal

By the end of the lab, aim to have one small working agent that uses at least one tool.